# **Crossref enrichement validation**

Este notebook importa e valida as funções de `src/bibliometric_rag_assistant/ingestion/crossref.py`, originadas da etapa exploratória no notebook `03_crossref_enrichement_exploration.ipynb`.

A execução lê as citações da última execução gravada no banco local e consulta a API do Crossref. Nada é gravado no banco nem em `data/raw/crossref`.
A persistência dos enriquecimentos é validada num banco SQLite em memória.

In [1]:
import json
import sqlite3
from contextlib import closing
from datetime import datetime, timezone

import httpx

from bibliometric_rag_assistant.config import CROSSREF_MAILTO, DB_PATH
from bibliometric_rag_assistant.legacy.db import (
    TABLES,
    connect_database,
    count_rows,
    create_tables,
    insert_citation_enrichments,
    insert_citations,
    insert_ingestion_run,
)
from bibliometric_rag_assistant.ingestion.crossref import (
    check_missing_dois,
    extract_crossref_year,
    fetch_crossref_work,
    fetch_crossref_works,
    parse_crossref_work,
    split_citations_by_doi,
)

CROSSREF_SAMPLE_SIZE = 5

print("Database:", DB_PATH)
print("Crossref mailto:", CROSSREF_MAILTO)

Database: /home/lucas/github/bibliometric-rag-assistant/data/interim/bibliometria.db
Crossref mailto: lucasalonso0096@gmail.com


### **Citations from the latest execution**

In [2]:
with closing(connect_database()) as connection:
    # Linhas acessíveis por nome de coluna, convertidas em dict abaixo
    connection.row_factory = sqlite3.Row

    rows = connection.execute(
        """
        SELECT run_id, pmid, doi, title, journal, publication_year
        FROM citations
        WHERE run_id = (
            SELECT run_id
            FROM ingestion_runs
            ORDER BY retrieved_at DESC
            LIMIT 1
        )
        ORDER BY pmid
        """
    ).fetchall()

citations = [dict(row) for row in rows]

assert citations, "Nenhuma citação no banco: rode scripts/legacy/ingest_sample.py antes"

citations[0]["run_id"], len(citations)

('20261023T114809Z', 20)

In [10]:
for i in range(len(rows)):
    print(rows[i]["run_id"])

20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z
20261023T114809Z


In [4]:
row = rows[0]
row[0], row["run_id"]

('20261023T114809Z', '20261023T114809Z')

### **Check missing DOIs**

Antes de qualquer chamada ao Crossref, o relatório mostra quantas e quais citações não têm DOI.

In [12]:
report = check_missing_dois(citations)

print(f"Total:   {report['total']}")
print(f"Com DOI: {report['with_doi']}")
print(f"Sem DOI: {report['without_doi']}")

report["missing"]

Total:   20
Com DOI: 20
Sem DOI: 0


[]

In [13]:
citations_with_doi = report["citations_with_doi"]

assert report["with_doi"] + report["without_doi"] == report["total"]
assert len(citations_with_doi) == report["with_doi"]
assert len(report["missing"]) == report["without_doi"]
assert all(citation["doi"] == citation["doi"].strip() for citation in citations_with_doi)
assert all(citation["doi"] for citation in citations_with_doi)

A amostra real pode não ter DOIs faltantes, então o caso é validado também com dados sintéticos: `None`, string vazia e só espaços contam como faltantes, e DOIs com espaços nas bordas são normalizados.

In [14]:
synthetic_citations = [
    {"pmid": "1", "doi": "10.1000/abc", "title": "Com DOI"},
    {"pmid": "2", "doi": "  10.1000/def  ", "title": "DOI com espaços"},
    {"pmid": "3", "doi": None, "title": "DOI nulo"},
    {"pmid": "4", "doi": "", "title": "DOI vazio"},
    {"pmid": "5", "doi": "   ", "title": "DOI só com espaços"},
    {"pmid": "6", "title": "Sem chave doi"},
]

with_doi, without_doi = split_citations_by_doi(synthetic_citations)

assert [citation["doi"] for citation in with_doi] == ["10.1000/abc", "10.1000/def"]
assert [citation["pmid"] for citation in without_doi] == ["3", "4", "5", "6"]

# A normalização não pode alterar a lista original
assert synthetic_citations[1]["doi"] == "  10.1000/def  "

synthetic_report = check_missing_dois(synthetic_citations)

assert synthetic_report["total"] == 6
assert synthetic_report["with_doi"] == 2
assert synthetic_report["without_doi"] == 4

synthetic_report["missing"]

[{'pmid': '3', 'title': 'DOI nulo'},
 {'pmid': '4', 'title': 'DOI vazio'},
 {'pmid': '5', 'title': 'DOI só com espaços'},
 {'pmid': '6', 'title': 'Sem chave doi'}]

In [15]:
split_citations_by_doi(synthetic_citations)

([{'pmid': '1', 'doi': '10.1000/abc', 'title': 'Com DOI'},
  {'pmid': '2', 'doi': '10.1000/def', 'title': 'DOI com espaços'}],
 [{'pmid': '3', 'doi': None, 'title': 'DOI nulo'},
  {'pmid': '4', 'doi': '', 'title': 'DOI vazio'},
  {'pmid': '5', 'doi': '   ', 'title': 'DOI só com espaços'},
  {'pmid': '6', 'title': 'Sem chave doi'}])

### **Consult a single DOI**

In [16]:
article = citations_with_doi[0]

crossref_response = fetch_crossref_work(article["doi"])

assert crossref_response["status"] == "ok"
assert crossref_response["message-type"] == "work"

# O Crossref devolve o DOI em minúsculas
assert crossref_response["message"]["DOI"].lower() == article["doi"].lower()

crossref_response.keys()

dict_keys(['status', 'message-type', 'message-version', 'message'])

### **Parsing for a single citation with DOI**

In [17]:
crossref_record = parse_crossref_work(crossref_response["message"])

assert set(crossref_record) == {
    "doi",
    "title",
    "journal",
    "publisher",
    "document_type",
    "publication_year",
    "publication_year_source",
    "published_print",
    "published_online",
    "volume",
    "issue",
    "page",
    "language",
    "issn",
    "issn_types",
    "authors",
    "author_details",
    "reference_count",
    "citation_count",
    "reference_dois",
}

for field in ["issn", "issn_types", "authors", "author_details", "reference_dois"]:
    assert isinstance(crossref_record[field], list), field

assert crossref_record["authors"] == [
    details["name"] for details in crossref_record["author_details"]
]

# reference_count conta todas as referências; nem todas têm DOI
assert len(crossref_record["reference_dois"]) <= crossref_record["reference_count"]

crossref_record

{'doi': '10.1111/odi.14597',
 'title': '3D‐printed temporomandibular joint‐mandible combined prosthesis: A prospective study',
 'journal': 'Oral Diseases',
 'publisher': 'Wiley',
 'document_type': 'journal-article',
 'publication_year': 2024,
 'publication_year_source': 'published-print',
 'published_print': '2024-04',
 'published_online': '2023-05-28',
 'volume': '30',
 'issue': '3',
 'page': '1360-1366',
 'language': 'en',
 'issn': ['1354-523X', '1601-0825'],
 'issn_types': [{'value': '1354-523X', 'type': 'print'},
  {'value': '1601-0825', 'type': 'electronic'}],
 'authors': ['Jisi Zheng',
  'Liang Huo',
  'Zixian Jiao',
  'Xiang Wei',
  'Lingtong Bu',
  'Wenbo Jiang',
  'Yi Luo',
  'Minjie Chen',
  'Chi Yang'],
 'author_details': [{'name': 'Jisi Zheng',
   'given': 'Jisi',
   'family': 'Zheng',
   'sequence': 'first',
   'orcid': None,
   'affiliations': ["Department of Oral Surgery, Shanghai Ninth People's Hospital Shanghai Jiao Tong University School of Medicine  Shanghai China",


Comparação lado a lado com o registro do PubMed para a mesma citação.

In [9]:
{
    field: {
        "pubmed": article.get(field),
        "crossref": crossref_record.get(field),
    }
    for field in ["doi", "title", "journal", "publication_year"]
}

{'doi': {'pubmed': '10.1111/odi.14597', 'crossref': '10.1111/odi.14597'},
 'title': {'pubmed': '3D-printed temporomandibular joint-mandible combined prosthesis: A prospective study.',
  'crossref': '3D‐printed temporomandibular joint‐mandible combined prosthesis: A prospective study'},
 'journal': {'pubmed': 'Oral diseases', 'crossref': 'Oral Diseases'},
 'publication_year': {'pubmed': 2024, 'crossref': 2024}}

O parser também precisa lidar com autores institucionais (campo `name`), datas ausentes e a prioridade entre os campos de data.

In [18]:
synthetic_work = {
    "DOI": "10.1000/xyz",
    "title": [],
    "author": [
        {
            "given": "Ana",
            "family": "Silva",
            "sequence": "first",
            "ORCID": "https://orcid.org/0000-0000-0000-0000",
            "affiliation": [{"name": "UFF"}, {}],
        },
        {"family": "Souza", "sequence": "additional"},
        {"name": "TMJ Study Group", "sequence": "additional"},
        {},
    ],
    "published-online": {"date-parts": [[2021, 5, 3]]},
    "published-print": {"date-parts": [[2022, 1]]},
    "issued": {"date-parts": [[2021]]},
    "issn-type": [{"value": "1234-5678", "type": "print"}],
    "reference": [
        {"key": "ref1", "DOI": "10.1000/ref1"},
        {"key": "ref2", "unstructured": "Referência sem DOI"},
    ],
}

synthetic_record = parse_crossref_work(synthetic_work)

assert synthetic_record["title"] is None
assert synthetic_record["journal"] is None
assert synthetic_record["volume"] is None
assert synthetic_record["issn"] == []
assert synthetic_record["issn_types"] == [{"value": "1234-5678", "type": "print"}]
assert synthetic_record["authors"] == ["Ana Silva", "Souza", "TMJ Study Group"]
assert synthetic_record["reference_dois"] == ["10.1000/ref1"]

first_author = synthetic_record["author_details"][0]

assert first_author["orcid"] == "https://orcid.org/0000-0000-0000-0000"
assert first_author["affiliations"] == ["UFF"]
assert synthetic_record["author_details"][2]["given"] is None

# Datas parciais preservam só a precisão disponível
assert synthetic_record["published_print"] == "2022-01"
assert synthetic_record["published_online"] == "2021-05-03"

# published-print tem prioridade sobre published-online
assert synthetic_record["publication_year"] == 2022
assert synthetic_record["publication_year_source"] == "published-print"

assert extract_crossref_year({}) == (None, None)
assert extract_crossref_year({"issued": {"date-parts": [[None]]}}) == (None, None)
assert extract_crossref_year({"issued": {"date-parts": [[2019]]}}) == (2019, "issued")

synthetic_record

{'doi': '10.1000/xyz',
 'title': None,
 'journal': None,
 'publisher': None,
 'document_type': None,
 'publication_year': 2022,
 'publication_year_source': 'published-print',
 'published_print': '2022-01',
 'published_online': '2021-05-03',
 'volume': None,
 'issue': None,
 'page': None,
 'language': None,
 'issn': [],
 'issn_types': [{'value': '1234-5678', 'type': 'print'}],
 'authors': ['Ana Silva', 'Souza', 'TMJ Study Group'],
 'author_details': [{'name': 'Ana Silva',
   'given': 'Ana',
   'family': 'Silva',
   'sequence': 'first',
   'orcid': 'https://orcid.org/0000-0000-0000-0000',
   'affiliations': ['UFF']},
  {'name': 'Souza',
   'given': None,
   'family': 'Souza',
   'sequence': 'additional',
   'orcid': None,
   'affiliations': []},
  {'name': 'TMJ Study Group',
   'given': None,
   'family': None,
   'sequence': 'additional',
   'orcid': None,
   'affiliations': []}],
 'reference_count': None,
 'citation_count': None,
 'reference_dois': ['10.1000/ref1']}

Entidades HTML em textos livres (`&amp;`, `&#227;`) são decodificadas pelo parser.

In [19]:
escaped_record = parse_crossref_work(
    {
        "title": ["Bone &amp; joint"],
        "container-title": ["Oral &amp; Maxillofacial Surgery"],
        "publisher": "A &amp; B",
        "author": [
            {
                "given": "Jo&#227;o",
                "family": "Leal",
                "affiliation": [{"name": "Lab A &amp; B"}],
            }
        ],
    }
)

assert escaped_record["title"] == "Bone & joint"
assert escaped_record["journal"] == "Oral & Maxillofacial Surgery"
assert escaped_record["publisher"] == "A & B"
assert escaped_record["authors"] == ["João Leal"]
assert escaped_record["author_details"][0]["affiliations"] == ["Lab A & B"]

### **Consulta em lote**

O fetcher em lote recebe somente `citations_with_doi`. Para não sobrecarregar a API, a validação usa uma amostra pequena.

In [20]:
sample = citations_with_doi[:CROSSREF_SAMPLE_SIZE]

responses, not_found = fetch_crossref_works(sample)

assert len(responses) + len(not_found) == len(sample)
assert all(response["status"] == "ok" for response in responses.values())

records = [parse_crossref_work(response["message"]) for response in responses.values()]

print(f"Encontrados: {len(responses)}")
print(f"Não encontrados: {not_found}")

[(record["doi"], record["publication_year"], record["citation_count"]) for record in records]

Encontrados: 5
Não encontrados: []


[('10.1111/odi.14597', 2024, 6),
 ('10.1007/s10006-023-01184-0', 2023, 0),
 ('10.1111/joor.13635', 2024, 18),
 ('10.1016/j.bjoms.2024.03.003', 2024, 1),
 ('10.1097/scs.0000000000010223', 2024, 6)]

Um DOI inexistente no Crossref deve gerar 404 na consulta individual e ir para `not_found` na consulta em lote, sem interromper as demais.

In [21]:
missing_doi = "10.0000/doi-inexistente-bibliometric-rag-assistant"

try:
    fetch_crossref_work(missing_doi)
except httpx.HTTPStatusError as error:
    assert error.response.status_code == 404
    print("Rejeitado como esperado:", error.response.status_code)
else:
    raise AssertionError("O DOI inexistente deveria ter retornado 404")

responses, not_found = fetch_crossref_works(
    [{"pmid": "0", "doi": missing_doi}, article]
)

assert not_found == [missing_doi]
assert list(responses) == [article["doi"]]

Rejeitado como esperado: 404


### **Persistência dos enriquecimentos**

Uma execução e três citações mínimas satisfazem as chaves estrangeiras. Cada citação recebe um dos três status possíveis.

In [14]:
memory = connect_database(":memory:")
create_tables(memory)

assert "citation_enrichments" in TABLES
assert count_rows(memory) == {table: 0 for table in TABLES}

insert_ingestion_run(
    memory,
    {
        "run_id": "run-test",
        "source": "pubmed",
        "query_text": "test",
        "query_variant": "test",
        "query_version": "0",
        "retrieved_at": "2026-01-01T00:00:00+00:00",
        "total_results": 3,
        "total_retrieved": 3,
    },
)
insert_citations(memory, "run-test", [{"pmid": "1"}, {"pmid": "2"}, {"pmid": "3"}])

retrieved_at = datetime.now(timezone.utc).isoformat()
base = {"run_id": "run-test", "source": "crossref", "retrieved_at": retrieved_at}

enrichments = [
    {
        **base,
        "pmid": "1",
        "doi": article["doi"],
        "status": "success",
        "raw": crossref_response,
        "normalized": crossref_record,
    },
    {**base, "pmid": "2", "doi": None, "status": "missing_doi"},
    {**base, "pmid": "3", "doi": missing_doi, "status": "not_found"},
]

insert_citation_enrichments(memory, enrichments)
memory.commit()

count_rows(memory)

{'ingestion_runs': 1,
 'raw_snapshots': 0,
 'citations': 3,
 'citation_enrichments': 3}

Os JSONs devem voltar iguais aos originais, e os status sem resposta gravam `NULL`.

In [15]:
raw_json, normalized_json = memory.execute(
    "SELECT raw_json, normalized_json FROM citation_enrichments WHERE status = 'success'"
).fetchone()

assert json.loads(raw_json) == crossref_response
assert json.loads(normalized_json) == crossref_record

assert memory.execute(
    """
    SELECT raw_json, normalized_json
    FROM citation_enrichments
    WHERE status <> 'success'
    """
).fetchall() == [(None, None), (None, None)]

Status fora do `CHECK`, citação inexistente e enriquecimento duplicado devem ser rejeitados.

In [16]:
invalid_enrichments = {
    "status inválido": {**enrichments[1], "source": "test", "status": "error"},
    "citação inexistente": {**enrichments[1], "pmid": "999"},
    "enriquecimento duplicado": enrichments[0],
}

for case, enrichment in invalid_enrichments.items():
    try:
        insert_citation_enrichments(memory, [enrichment])
    except sqlite3.IntegrityError as error:
        print(f"{case}: rejeitado como esperado ({error})")
    else:
        raise AssertionError(f"{case}: deveria ter sido rejeitado")

memory.rollback()

assert count_rows(memory)["citation_enrichments"] == len(enrichments)

status inválido: rejeitado como esperado (CHECK constraint failed: status IN (
                    'success',
                    'missing_doi',
                    'not_found'
                ))
citação inexistente: rejeitado como esperado (FOREIGN KEY constraint failed)
enriquecimento duplicado: rejeitado como esperado (UNIQUE constraint failed: citation_enrichments.run_id, citation_enrichments.pmid, citation_enrichments.source)


In [17]:
memory.close()